In [4]:
from pathlib import Path
import pandas as pd

# Prefer the exact dataset file in the current notebook folder.
target_file = Path("Sentence pairs in Japanese-English - 2026-03-24.tsv")

if target_file.exists():
    dataset_path = target_file
else:
    # Fallback: search common roots for sentence-pair datasets.
    search_roots = [Path("."), Path("/kaggle/input")]
    candidates = []

    for root in search_roots:
        if root.exists():
            for p in root.rglob("*"):
                if p.is_file():
                    name = p.name.lower()
                    if "sentence" in name and "pair" in name and p.suffix.lower() in {".csv", ".tsv", ".json", ".parquet"}:
                        candidates.append(p)

    if not candidates:
        raise FileNotFoundError("No dataset file found with both 'sentence' and 'pair' in its filename.")

    dataset_path = sorted(candidates)[0]

print(f"Using dataset: {dataset_path}")

# Read dataset based on extension with consistent column names.
suffix = dataset_path.suffix.lower()
if suffix in {".csv", ".tsv"}:
    df = pd.read_csv(
        dataset_path,
        sep="\t" if suffix == ".tsv" else ",",
        header=None,
        names=['eng_id', 'english', 'jp_id', 'japanese'],
    )
elif suffix == ".json":
    df = pd.read_json(dataset_path)
elif suffix == ".parquet":
    df = pd.read_parquet(dataset_path)
else:
    raise ValueError(f"Unsupported file type: {suffix}")

df.head(5)

Using dataset: Sentence pairs in English-Japanese - 2026-03-24.tsv


,eng_id,english,jp_id,japanese
0,1276,Let's try something.,4702,何かしてみましょう。
1,1277,I have to go to sleep.,4703,私は眠らなければなりません。
2,1277,I have to go to sleep.,344904,そろそろ寝なくちゃ。
3,1280,Today is June 18th and it is Muiriel's birthday!,4705,今日は６月１８日で、ムーリエルの誕生日です！
4,1282,Muiriel is 20 now.,4707,ムーリエルは２０歳になりました。


In [2]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279694 entries, 0 to 279693
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   sentence_id_1  279694 non-null  int64 
 1   sentence_1     279694 non-null  object
 2   sentence_id_2  279694 non-null  int64 
 3   sentence_2     279694 non-null  object
dtypes: int64(2), object(2)
memory usage: 8.5+ MB


,sentence_id_1,sentence_id_2
count,2.796940e+05,2.796940e+05
mean,2.454729e+06,3.025899e+06
std,3.767512e+06,4.403922e+06
min,1.276000e+03,1.297000e+03
25%,2.412132e+05,1.476042e+05
50%,3.092180e+05,2.107650e+05
75%,2.973504e+06,6.850130e+06
max,1.381550e+07,1.381093e+07


In [5]:
df.head(5)

,eng_id,english,jp_id,japanese
0,1276,Let's try something.,4702,何かしてみましょう。
1,1277,I have to go to sleep.,4703,私は眠らなければなりません。
2,1277,I have to go to sleep.,344904,そろそろ寝なくちゃ。
3,1280,Today is June 18th and it is Muiriel's birthday!,4705,今日は６月１８日で、ムーリエルの誕生日です！
4,1282,Muiriel is 20 now.,4707,ムーリエルは２０歳になりました。


## PDF Chunking for RAG Knowledge Base

This section processes two PDF documents into chunked `.jsonl` files for a RAG knowledge base:

1. **JTF Style Guide** (`jtf_style_guide_e.pdf`) — chunked by numbered section headings
2. **Tae Kim's Grammar Guide** (`grammar_guide.pdf`) — selected sections only: particle usage (は/も/が), polite vs plain form, transitive/intransitive verbs

Each chunk is a dict with fields: `source`, `section`, `text`, `chunk_id`. Outputs are saved to `data/kb/`.

In [2]:
%pip install -q pdfplumber

import pdfplumber
import json
import re
from pathlib import Path

KB_DIR = Path("data/kb")
KB_DIR.mkdir(parents=True, exist_ok=True)


def extract_pdf_text(pdf_path):
    """Extract text from every page of a PDF, returned as one string."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n".join(pages)


def clean_pdf_text(text):
    """Clean common PDF extraction artifacts.

    Full-width numbers/letters are preserved — in these documents
    they are intentional examples, not extraction errors.
    """
    text = re.sub(r"--\s*\d+\s+of\s+\d+\s*--", "", text)
    text = text.replace("\ufb01", "fi").replace("\ufb02", "fl")
    text = text.replace("\u2018", "'").replace("\u2019", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    text = text.replace("\u2013", "-").replace("\u2014", "--")
    text = text.replace("\x0c", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"\n\d{1,3}\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def split_sentences(text):
    """Split mixed EN/JP text into sentence-level units."""
    paragraphs = re.split(r"\n\s*\n", text.strip())
    sentences = []
    for para in paragraphs:
        para = para.strip()
        if not para:
            continue
        sents = re.split(
            r"(?<=[。！？])\s*(?=\S)|(?<=[.!?])\s+(?=[A-Z\"'(])",
            para,
        )
        sentences.extend(s.strip() for s in sents if s and s.strip())
    return sentences


def chunk_with_overlap(sentences, max_sents=5, overlap=2):
    """Group sentences into overlapping chunks."""
    if not sentences:
        return []
    if len(sentences) <= max_sents:
        return [" ".join(sentences)]
    chunks, step = [], max(1, max_sents - overlap)
    for i in range(0, len(sentences), step):
        chunk_sents = sentences[i : i + max_sents]
        chunks.append(" ".join(chunk_sents))
        if i + max_sents >= len(sentences):
            break
    return chunks


def estimate_tokens(text):
    """Rough token count (~4 chars / token for mixed EN/JP)."""
    return max(1, len(text) // 4)


def save_jsonl(records, path):
    """Write list of dicts as a .jsonl file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} chunks → {path}")


def print_summary(chunks, label):
    """Print chunk count, average token length, and sample chunks."""
    toks = [estimate_tokens(c["text"]) for c in chunks]
    avg = sum(toks) / len(toks) if toks else 0
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"  Total chunks : {len(chunks)}")
    print(f"  Avg length   : {avg:.0f} tokens (est.)\n")
    for c in chunks[:3]:
        preview = c["text"][:300] + ("…" if len(c["text"]) > 300 else "")
        print(f"  [{c['chunk_id']}]  section: {c['section']}")
        print(f"  {preview}\n")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 1. JTF Style Guide (`jtf_style_guide_e.pdf`)

Extract text with `pdfplumber`, split by numbered section headings (e.g. "1.1", "2.1.6"), and create sentence-level chunks with 1–2 sentence overlap. Each section becomes one or more chunks depending on its length.

In [3]:
raw_style = extract_pdf_text("data/jtf_style_guide_e.pdf")
style_text = clean_pdf_text(raw_style)

# Match section headings like "1.1. Style", "2.1.6. Chōon ...", etc.
# Requires at least one dot (e.g. "1.1") to avoid matching numbered list items.
heading_re = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.+?)$", re.MULTILINE)
matches = list(heading_re.finditer(style_text))
print(f"Found {len(matches)} section headings in JTF style guide\n")

style_chunks = []
source_style = "jtf_style_guide_e.pdf"

for idx, m in enumerate(matches):
    sec_num = m.group(1)
    sec_title = m.group(2).strip()
    heading = f"{sec_num} {sec_title}"

    start = m.start()
    end = matches[idx + 1].start() if idx + 1 < len(matches) else len(style_text)
    body = style_text[start:end].strip()

    sents = split_sentences(body)
    sub_chunks = chunk_with_overlap(sents, max_sents=5, overlap=2)

    for j, ct in enumerate(sub_chunks):
        style_chunks.append({
            "source": source_style,
            "section": heading,
            "text": ct,
            "chunk_id": f"jtf_{sec_num.replace('.', '_')}_{j:02d}",
        })

save_jsonl(style_chunks, KB_DIR / "style_guide_chunks.jsonl")
print_summary(style_chunks, "JTF Style Guide — Chunking Summary")

Found 143 section headings in JTF style guide

Saved 181 chunks → data\kb\style_guide_chunks.jsonl

  JTF Style Guide — Chunking Summary
  Total chunks : 181
  Avg length   : 63 tokens (est.)

  [jtf_12_5_00]  section: 12.5 12．5
  12.5 12．5
single-byte comma (,)
12. Be consistent in unit nomenclature. Acceptable Unacceptable See
2kg 354m 20℃ 3ft 23坪 14km 5. Writing Units
Is consistently metric. Mixes American, Japanese, and
metric units.

  [jtf_12_5_01]  section: 12.5 12．5
  Writing Units
Is consistently metric. Mixes American, Japanese, and
metric units. Table of Contents
Introduction
1. Sentences ···················································································· 9

  [jtf_1_1_00]  section: 1.1 Style ................................................................................................... 9
  1.1. Style ................................................................................................... 9



### 2. Tae Kim's Grammar Guide (`grammar_guide.pdf`) — Selected Sections

Extract only three targeted topics from the 353-page guide:

- **Particle usage (は/も/が)** — sections 3.3 through 3.3.4
- **Polite vs plain form (です/ます vs だ/である)** — sections 4.1 through 4.1.5
- **Transitive & intransitive verbs** — sections 3.9 through 3.9.1

Same chunking strategy: sentence-level splits with 1–2 sentence overlap.

In [4]:
TARGET_SECTIONS = {
    "3.3", "3.3.1", "3.3.2", "3.3.3", "3.3.4",      # Particles は/も/が
    "3.9", "3.9.1",                                     # Transitive / intransitive
    "4.1", "4.1.1", "4.1.2", "4.1.3", "4.1.4", "4.1.5", # Polite form
}

raw_gram = extract_pdf_text("data/grammar_guide.pdf")
gram_text = clean_pdf_text(raw_gram)

# Remove repeating page headers like "3.3. INTRO TO PARTICLES CHAPTER 3. BASIC GRAMMAR"
# These contain section numbers that would confuse heading detection.
gram_text = re.sub(r"^.*CHAPTER \d+\..*$", "", gram_text, flags=re.MULTILINE)

heading_re = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.+?)$", re.MULTILINE)
all_matches = list(heading_re.finditer(gram_text))
print(f"Found {len(all_matches)} total section headings in grammar guide")

grammar_chunks = []
source_gram = "grammar_guide.pdf"
kept = 0

for idx, m in enumerate(all_matches):
    sec_num = m.group(1)
    if sec_num not in TARGET_SECTIONS:
        continue
    kept += 1
    sec_title = m.group(2).strip()
    heading = f"{sec_num} {sec_title}"

    start = m.start()
    end = all_matches[idx + 1].start() if idx + 1 < len(all_matches) else len(gram_text)
    body = gram_text[start:end].strip()

    sents = split_sentences(body)
    sub_chunks = chunk_with_overlap(sents, max_sents=5, overlap=2)

    for j, ct in enumerate(sub_chunks):
        grammar_chunks.append({
            "source": source_gram,
            "section": heading,
            "text": ct,
            "chunk_id": f"gram_{sec_num.replace('.', '_')}_{j:02d}",
        })

print(f"Kept {kept} sections matching target topics\n")
save_jsonl(grammar_chunks, KB_DIR / "grammar_chunks.jsonl")
print_summary(grammar_chunks, "Grammar Guide (Selected Sections) — Chunking Summary")

Found 570 total section headings in grammar guide
Kept 26 sections matching target topics

Saved 81 chunks → data\kb\grammar_chunks.jsonl

  Grammar Guide (Selected Sections) — Chunking Summary
  Total chunks : 81
  Avg length   : 97 tokens (est.)

  [gram_3_3_00]  section: 3.3 Introduction to Particles . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 33
  3.3 Introduction to Particles . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 33

  [gram_3_3_1_00]  section: 3.3.1 Defining grammatical functions with particles . . . . . . . . . . . . . . . . 33
  3.3.1 Defining grammatical functions with particles . . . . . . . . . . . . . . . . 33

  [gram_3_3_2_00]  section: 3.3.2 The 「は」 topic particle . . . . . . . . . . . . . . . . . . . . . . . . . . . 33
  3.3.2 The 「は」 topic particle . . . . . . . . . . . . . . . . . . . . . . . . . . . 33

